# 데이터모두병합.ipynb와 과정은 같으나 들락날락 위치 데이터를 군집화한 위치 데이터로 변경했습니다.

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import math

# 등고선 데이터(라인) 읽기 및 좌표계 변환 (EPSG:4326)
contour_gdf = gpd.read_file("../부산광역시_등고선/부산광역시_등고선_전체.gpkg").to_crs(epsg=4326)

# 위치 정보가 담긴 CSV 파일을 읽어 DataFrame 생성
# 파일은 clustering_최적위치선정/README.md에서 첨부한 링크를 통해 다운로드
# 실제 파일 경로로 변경
gdf = gpd.read_file('clusters_gdf_all_gu.shp')

# 각 위치별로 가장 가까운 등고선의 등고수치(고도) 구하기
def find_nearest_contour(point, contour_gdf):
    distances = contour_gdf.geometry.distance(point)
    idx_min = distances.idxmin()
    return contour_gdf.loc[idx_min, '등고수치']

gdf['contour'] = gdf.geometry.apply(lambda x: find_nearest_contour(x, contour_gdf))

# 거리 계산을 위해 투영 좌표계(EPSG:5179)로 변환
contour_gdf_proj = contour_gdf.to_crs(epsg=5179)
gdf_proj = gdf.to_crs(epsg=5179)

# 각 위치별로 반경 540m 내 등고선의 등고수치 최소/최대값 및 고도차 계산
gdf['contour_min_540'] = None
gdf['contour_max_540'] = None
gdf['high_up'] = None    # 현재 위치 고도 - 540m 내 등고수치 최소값
gdf['high_down'] = None  # 540m 내 등고수치 최대값 - 현재 위치 고도

for idx, row in gdf_proj.iterrows():
    buffer = row.geometry.buffer(540)
    intersected = contour_gdf_proj[contour_gdf_proj.intersects(buffer)]
    if not intersected.empty:
        z_min = intersected['등고수치'].min()
        z_max = intersected['등고수치'].max()
        z_cur = gdf.loc[idx, 'contour']
        high_up = z_cur - z_min
        high_down = z_max - z_cur
    else:
        z_min, z_max, high_up, high_down = None, None, None, None

    gdf.at[idx, 'contour_min_540'] = z_min
    gdf.at[idx, 'contour_max_540'] = z_max
    gdf.at[idx, 'high_up'] = high_up
    gdf.at[idx, 'high_down'] = high_down

# 시군구별 0~12세 인구 비율이 포함된 행정경계 데이터 읽기
sgg = gpd.read_file('../아동인구수전처리/시군구별 0~12세 비율.shp')

# 좌표계 통일 (EPSG:4326)
points = gdf.to_crs(epsg=4326)
sgg = sgg.to_crs(epsg=4326)

# 각 위치(Point)가 어느 시군구(Polygon)에 포함되는지 공간 조인하여 시군구명과 아동인구수비율 정보 추가
joined = gpd.sjoin(points, sgg[['SGG_NM', 'geometry', '0~12세_비']], how='left', predicate='within')

# 불필요한 컬럼 제거 및 컬럼명 한글로 변경
joined.drop(['index_right'], axis=1, inplace=True)
joined = joined.rename(columns={'SGG_NM': '시군구', '0~12세_비': '어린이비율'})

# 컬럼 순서 재정렬
new_columns = ['cluster_id', 'data_point', '시군구', 'geometry', '어린이비율', 'contour', 'contour_min_540', 'contour_max_540', 'high_up', 'high_down']
joined = joined[new_columns]

# 결과를 shp 파일로 저장
joined.to_file('위치+등고선+시군구_비율.shp', encoding='utf-8')

# 등고수치 기준 내림차순으로 정렬하고 결과 확인
joined_sorted = joined.sort_values(by='contour', ascending=False)
joined_sorted.head(10)